# Voting

In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score, log_loss

from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.ensemble import VotingClassifier, VotingRegressor

from tqdm import tqdm  # Provides the progress of model running

In [4]:
kyph = pd.read_csv("D:/Machine_Learning/Cases/Kyphosis/Kyphosis.csv")
X , y = kyph.drop('Kyphosis', axis=1), kyph['Kyphosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify=y)

In [5]:
dtc = DecisionTreeClassifier(max_depth=4, random_state=26)
knn = KNeighborsClassifier(n_neighbors=3)
lr = LogisticRegression()
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()

voting = VotingClassifier(estimators= [('Tree', dtc),('KNN', knn), ('LR', lr), ('LDA', lda), ('NB', nb)])  # by default voting = hard
voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)
accuracy_score(y_test, y_pred)

0.88

In [6]:
dtc = DecisionTreeClassifier(max_depth=4, random_state=26)
knn = KNeighborsClassifier(n_neighbors=3)
lr = LogisticRegression()
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()

voting = VotingClassifier(estimators= [('Tree', dtc),('KNN', knn), ('LR', lr), ('LDA', lda), ('NB', nb)], voting='soft')  
voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)
accuracy_score(y_test, y_pred)

0.8

In [9]:
voting = VotingClassifier(estimators= [('Tree', dtc),('KNN', knn), ('LR', lr), ('LDA', lda), ('NB', nb)], voting='soft') 
voting.fit(X_train, y_train)
y_pred_prob = voting.predict_proba(X_test)
log_loss(y_test, y_pred_prob)

0.43740947651061596

In [52]:
voting = VotingClassifier(estimators= [('Tree', dtc),('KNN', knn), ('LR', lr), ('LDA', lda), ('NB', nb)], 
                          voting='soft', weights=[0.5, 0.01, 5, 7, 3])   # updates weights to lower logloss
voting.fit(X_train, y_train)
y_pred_prob = voting.predict_proba(X_test)
log_loss(y_test, y_pred_prob)

0.3491345803645122

In [23]:
estims = [dtc, knn, lr, lda, nb]
for e in estims:
    e.fit(X_train, y_train)
    y_pred_prob = e.predict_proba(X_test)
    print("Log_Loss of", e, "=", log_loss(y_test, y_pred_prob) )

Log_Loss of DecisionTreeClassifier(max_depth=4, random_state=26) = 4.582748472683514
Log_Loss of KNeighborsClassifier(n_neighbors=3) = 6.0123482471692045
Log_Loss of LogisticRegression() = 0.3357708471894506
Log_Loss of LinearDiscriminantAnalysis() = 0.34281059610597553
Log_Loss of GaussianNB() = 0.39698492604441094


# Boston Data

In [61]:
boston = pd.read_csv("D:/Machine_Learning/Datasets/Boston.csv")
X , y = boston.drop('medv', axis=1), boston['medv']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26)

In [72]:
lr = LinearRegression()
est = ElasticNet()
dtr = DecisionTreeRegressor(random_state=26)

voting = VotingRegressor(estimators= [('LR', lr),('EST', est), ('DTC', dtr)], weights=[7, 6, 20])
voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)
r2_score(y_test, y_pred)

0.866083958076251